# Prueba en Colab L4: Las 5 Mejoras del Tracer WSI sobre `LiquidAI/LFM3-1.7B`

Valida sobre el checkpoint real de `LFM3-1.7B` y el `.moipatch` guardado en Drive (`lfm3_safety_routed_refusal.moipatch`), usando **únicamente prompts benignos de OR-Bench + controles sanos** (cero texto tóxico):
1. **Linealización exacta de la norma final + gaps por bloque (`attribute_direction_writers`):** elimina el gap residual post-norm (~49 -> ~0) y localiza los gaps internos (como el de `L22`).
2. **Trazado dual / dirección protegida (`protect_dir=veto_dir`):** compara `plain` vs `protected` (`linear` y `fisher`) separando sitios que mueven `refusal_dir` sin dañar `veto_dir`.
3. **`logit_margin` vs `token_nll` saturado:** evita gradientes casi nulos cuando `P(token) -> 1`.
4. **Grafo causal multi-salto (`trace_direction_graph`) + diagnóstico benigno de framing educativo sobre `veto_dir`:** extrae rutas dirigidas para `refusal_dir` y `veto_dir`, y explica qué tokens y módulos causan la caída del veto ante el prefijo educativo.
5. **Peeling iterativo Hydra (`peel_causal_circuit`):** 3 rondas con y sin `protect_dir`, comprobando restauración bit a bit (`restore_exact_ok`) y huella SHA-256 (`fingerprint_ok`).

In [ ]:
MODEL = "LiquidAI/LFM3-1.7B"
RUN_NAME = "tracer_upgrades_lfm3_l4"
REPO = "/content/wsi"
BASE = "/content/base"
PATCH_DRIVE = "/content/drive/MyDrive/wsi_runs/surgical_refusal_lfm3/lfm3_safety_routed_refusal.moipatch"
N_PROMPTS = 4
PEEL_ALPHA = 6.0

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q torch transformers accelerate safetensors pandas pyarrow huggingface_hub datasets
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__)

In [ ]:
import io, os, shutil, subprocess, zipfile
from google.colab import drive, files

drive.mount("/content/drive")
FINALES = "/content/drive/MyDrive/wsi_runs/FINALES"
RUN_DIR = "/content/drive/MyDrive/wsi_runs/surgical_refusal_lfm3"
os.makedirs(FINALES, exist_ok=True)
os.makedirs(RUN_DIR, exist_ok=True)

if not os.path.exists(PATCH_DRIVE):
    raise FileNotFoundError(f"No encuentro el .moipatch de seguridad en Drive: {PATCH_DRIVE}")

if not os.path.exists(REPO):
    !git clone -q --depth 1 https://github.com/krexito/weight-space-index.git {REPO}
dirty = subprocess.check_output(["git", "-C", REPO, "status", "--porcelain"], text=True).strip()
if not dirty:
    subprocess.run(["git", "-C", REPO, "fetch", "--depth", "1", "origin", "main"], check=True)
    subprocess.run(["git", "-C", REPO, "reset", "--hard", "origin/main"], check=True)
COMMIT = subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"], text=True).strip()

def _missing_files():
    tp = f"{REPO}/wsi/tracer.py"
    sp = f"{REPO}/scripts/probe_tracer_upgrades_lfm3.py"
    missing = []
    if not os.path.exists(tp):
        missing.append("wsi/tracer.py")
    else:
        txt = open(tp, encoding="utf-8").read()
        if not all(sym in txt for sym in ("trace_direction_graph", "peel_causal_circuit", "protect_dir", "logit_margin", "_linearize_final_norm_span")):
            missing.append("wsi/tracer.py (versión local)")
    if not os.path.exists(sp):
        missing.append("scripts/probe_tracer_upgrades_lfm3.py")
    return missing

dest_map = {
    "tracer.py": f"{REPO}/wsi/tracer.py",
    "__init__.py": f"{REPO}/wsi/__init__.py",
    "mcp_server.py": f"{REPO}/wsi/mcp_server.py",
    "probe_tracer_upgrades_lfm3.py": f"{REPO}/scripts/probe_tracer_upgrades_lfm3.py",
}
for _attempt in range(3):
    missing = _missing_files()
    if not missing:
        if _attempt == 0:
            print("Mejoras del tracer detectadas en", REPO)
        break
    print(f"Modo pre-commit (faltan: {missing}). Sube los archivos o un .zip con ellos (puedes hacerlo en 1-2 pasos):")
    uploaded = files.upload()
    for up_name, raw in uploaded.items():
        if up_name.endswith(".zip"):
            with zipfile.ZipFile(io.BytesIO(raw)) as zf:
                for member in zf.namelist():
                    base = os.path.basename(member)
                    if base in dest_map and not member.endswith("/"):
                        os.makedirs(os.path.dirname(dest_map[base]), exist_ok=True)
                        with open(dest_map[base], "wb") as f:
                            f.write(zf.read(member))
                        print("Extraído:", dest_map[base])
        else:
            base = os.path.basename(up_name)
            if base in dest_map:
                os.makedirs(os.path.dirname(dest_map[base]), exist_ok=True)
                shutil.move(up_name, dest_map[base])
                print("Copiado:", dest_map[base])
else:
    still = _missing_files()
    if still:
        raise RuntimeError(f"Siguen faltando archivos en {REPO}: {still}")

if not os.path.exists(BASE):
    from huggingface_hub import snapshot_download
    snapshot_download(MODEL, local_dir=BASE)
print("commit:", COMMIT, "| patch:", PATCH_DRIVE)

In [ ]:
import os, subprocess, sys

LOCAL_OUT = f"/content/{RUN_NAME}.json"
SCRIPT = f"{REPO}/scripts/probe_tracer_upgrades_lfm3.py"
if not os.path.exists(SCRIPT):
    raise FileNotFoundError(f"Falta {SCRIPT}")

cmd = [
    sys.executable, "-u", SCRIPT,
    "--model", BASE,
    "--patch", PATCH_DRIVE,
    "--n-prompts", str(N_PROMPTS),
    "--peel-alpha", str(PEEL_ALPHA),
    "--out", LOCAL_OUT,
]
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end="", flush=True)
rc = proc.wait()
if rc != 0:
    raise subprocess.CalledProcessError(rc, cmd)

In [ ]:
import datetime, json, os, subprocess

res = json.load(open(LOCAL_OUT, encoding="utf-8"))
print("=== RESUMEN DE LAS 5 MEJORAS DEL TRACER (LiquidAI/LFM3-1.7B) ===")
p1 = res["probe1_linearization_and_gaps"]
pn = p1["post_final_norm"]
print(f"1) Post-norm L{pn['target_layer']} residual_gap: sin_lin={pn['unlinearized_residual_gap']} -> con_lin={pn['linearized_residual_gap']} (ok={pn['final_norm_linearized']})")
print("   Top block gaps (refusal):", [(g["layer"], g["gap"]) for g in p1["router_layer_refusal"]["top_block_gaps"][:5]])
if "router_layer_veto" in p1:
    print("   Top block gaps (veto)   :", [(g["layer"], g["gap"]) for g in p1["router_layer_veto"]["top_block_gaps"][:5]])

p2 = res["probe2_protected_dir"]
print("\n2) Plain vs Protected (protect_dir=veto_dir):")
print("   Plain Top-5   :", [(s["module"], s["pareto_score"]) for s in p2["plain_linear"][:5]])
if "protected_linear" in p2:
    print("   Prot-Lin Top-5:", [(s["module"], s["pareto_score"], s["healthy_dnll"]) for s in p2["protected_linear"][:5]])
    print("   Prot-Fish Top-5:", [(s["module"], s["pareto_score"], s["healthy_dnll"]) for s in p2["protected_fisher"][:5]])

p3 = res["probe3_logit_margin_vs_nll"]
print(f"\n3) logit_margin vs token_nll (P(tok)={p3['eval_refusal_prob']}):")
print("   token_nll   sum|attr|:", p3["token_nll"]["sum_abs_causal_attr"], "| top3:", [(r["module"], r["causal_attr"]) for r in p3["token_nll"]["top_modules"][:3]])
print("   logit_margin sum|attr|:", p3["logit_margin"]["sum_abs_causal_attr"], "| score:", p3["logit_margin"]["target_obj_score"], "| top3:", [(r["module"], r["causal_attr"]) for r in p3["logit_margin"]["top_modules"][:3]])

p4 = res["probe4_direction_graph"]
rg = p4["refusal_graph"]
print(f"\n4) Grafo multi-salto refusal ({rg['n_nodes']} nodos, {rg['n_edges']} aristas):")
for tp in rg["top_paths"][:5]:
    print("  ", " -> ".join(tp["nodes"]), "| weights=", tp["edge_weights"], "| share=", tp["path_share"])
if "veto_graph" in p4:
    vg = p4["veto_graph"]
    print(f"   Grafo multi-salto veto ({vg['n_nodes']} nodos, {vg['n_edges']} aristas):")
    for tp in vg["top_paths"][:5]:
        print("  ", " -> ".join(tp["nodes"]), "| weights=", tp["edge_weights"], "| share=", tp["path_share"])
if "benign_edu_framing_veto_diagnostic" in p4:
    bd = p4["benign_edu_framing_veto_diagnostic"]
    print(f"\n   Diagnostico framing educativo sobre veto_dir: {bd['raw_veto_score']} -> {bd['edu_veto_score']} (delta={bd['delta_veto_score']}, embed={bd['delta_embed_contrib']})")
    print("   Top shift writers:", [(r["module"], r.get("delta_contrib")) for r in bd["top_shift_modules"][:5]])
    print("   Top shift heads  :", [(r["module"], r["head"], r.get("delta_contrib")) for r in bd["top_shift_heads"][:5]])
    print("   Top edu tokens   :", [(t.get("token"), round(t["grad_x_input"], 4)) for t in bd["top_input_tokens_edu0"][:8]])

p5 = res["probe5_hydra_peeling"]
print("\n5) Hydra peeling (3 rondas):")
pl = p5["plain"]
print("   Plain:", pl["selected_modules"], "| traj=", pl["score_trajectory"], "| restored=", pl["restore_exact_ok"], "| fp=", pl["fingerprint_ok"])
if "protected" in p5:
    pr = p5["protected"]
    print("   Prot :", pr["selected_modules"], "| traj=", pr["score_trajectory"], "| protect=", pr["protect_trajectory"], "| restored=", pr["restore_exact_ok"], "| fp=", pr["fingerprint_ok"])

res["_final"] = {
    "complete": True,
    "model": MODEL,
    "commit": COMMIT,
    "notebook": "probe_tracer_upgrades_lfm3_colab.ipynb",
    "gpu": subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True).strip(),
    "saved_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
}
for out_dir in (FINALES, RUN_DIR):
    os.makedirs(out_dir, exist_ok=True)
    dst = f"{out_dir}/{RUN_NAME}.json"
    with open(dst + ".tmp", "w", encoding="utf-8") as f:
        json.dump(res, f, indent=2)
    os.replace(dst + ".tmp", dst)
print("\nGuardado final en Drive:", f"{FINALES}/{RUN_NAME}.json")